# 🩺 Reinforcement Learning Lab - Automatic Blood Glucose Control

**Estimated duration: 3h**

## Objective

In this lab, you will train a **Reinforcement Learning (RL) agent** to automatically control insulin injection for a Type 1 diabetic patient. Unlike supervised learning where we have (input, expected output) pairs, here the agent must **learn by trial and error** by interacting with a patient simulator.

---

## Fundamental Difference with Supervised Learning

| Supervised Learning | Reinforcement Learning |
|---------------------|------------------------|
| Fixed dataset (X, y) | No dataset, the agent explores |
| The correct answer is known | Only a reward signal is known |
| Single training pass | Continuous agent ↔ environment loop |
| The model predicts | The agent acts and observes consequences |

In RL, the agent must:
1. **Observe** the current state (here: the patient's blood glucose)
2. **Choose an action** (here: the insulin dose to inject)
3. **Receive a reward** (here: based on the quality of glycemic control)
4. **Learn** to maximize the sum of future rewards

---

## Part 1: Medical Context and Simulator

### 1.1 Type 1 Diabetes

Type 1 diabetes is an autoimmune disease where the pancreas no longer produces insulin. Patients must therefore inject insulin to regulate their blood glucose (blood sugar level, noted **BG** for *Blood Glucose*).

**Important glycemic ranges:**
- **Hypoglycemia** (BG < 70 mg/dL): Dangerous! Can cause fainting, seizures, coma
- **Target zone** (70 ≤ BG ≤ 180 mg/dL): Safe zone, treatment goal
- **Hyperglycemia** (BG > 180 mg/dL): Harmful in the long term (complications)

**Important clinical metric: TIR (Time In Range)**
- Percentage of time spent in the target zone [70-180] mg/dL
- Clinical goal: TIR > 70%

### 1.2 The UVa/Padova Simulator (simglucose)

We use **simglucose**, an open-source simulator based on the **UVa/Padova** model, approved by the US FDA for *in silico* testing of glycemic control systems. [GitHub](https://github.com/jxx123/simglucose)

**How was this simulator created?**
- Based on physiological mathematical models (differential equations)
- Calibrated on real clinical data from diabetic patients
- Simulates glucose-insulin dynamics, meal absorption, inter-patient variability
- Contains 30 virtual patients (10 adults, 10 adolescents, 10 children)

**In this lab, we work with a single patient: `adult#001`**

---

## Part 2: Formalizing the RL Problem

### 2.1 Key Elements of Reinforcement Learning

Any RL problem is defined by:

1. **Observation Space (State Space)**: What the agent can perceive
2. **Action Space**: What the agent can do
3. **Reward Function**: The learning signal
4. **Environment Dynamics**: How the state evolves after an action

### 2.2 Our Formalization

**Observation:** Vector of past blood glucose readings at different time horizons
- Current BG (t=0)
- BG 3 min ago (t-1 step)
- BG 6 min ago (t-2 steps)
- BG 15 min ago (t-5 steps)
- BG 30 min ago (t-10 steps)
- BG 1h ago (t-20 steps)
- BG 2h ago (t-40 steps)
- BG 4h ago (t-80 steps)

Plus the corresponding insulin doses → **16 dimensions in total**

**Action:** Continuous basal insulin dose
- Range: [0, 0.1] U/min (insulin units per minute)
- Represented by a value in [-1, 1] then converted

**Reward:** Based on Kovatchev's risk index (see Part 5)


Reward = Risk(t-1) - Risk(t)
- If risk decreases → positive reward
- If risk increases → negative reward

**Q1.** In our formalization, why do we use a history of blood glucose readings rather than just the current blood glucose?

**Answer:**

**Q2.** The reward is defined as `reward = risk(t-1) - risk(t)`. Explain why this formulation encourages the agent to maintain stable blood glucose within the target zone.

**Answer:**

---

## Part 3: Installation and Imports

**Q3.** Run the following cell to install the required libraries.


In [ ]:
!pip install -q simglucose==0.2.11 stable-baselines3[extra] gymnasium matplotlib pandas

**Q4.** Import the following libraries:
- `numpy` (alias `np`)
- `matplotlib.pyplot` (alias `plt`)
- `pandas` (alias `pd`)
- `datetime` from the `datetime` module
- `deque` from the `collections` module

From simglucose:
- `T1DSimEnv` from `simglucose.simulation.env`
- `T1DPatient` from `simglucose.patient.t1dpatient`
- `CGMSensor` from `simglucose.sensor.cgm`
- `InsulinPump` from `simglucose.actuator.pump`
- `RandomScenario` from `simglucose.simulation.scenario_gen`
- `Action` from `simglucose.controller.base`

From gymnasium and stable-baselines3:
- `gymnasium` (alias `gym`)
- `spaces` from `gymnasium`
- `PPO` from `stable_baselines3`



In [ ]:
# YOUR CODE HERE


---

## Part 4: Understanding Stable-Baselines3

### 4.1 What is Stable-Baselines3?

**Stable-Baselines3 (SB3)** is a Python library that implements Reinforcement Learning algorithms in a reliable and optimized way. It provides:
- State-of-the-art algorithm implementations (PPO, SAC, DQN, A2C, etc.)
- A unified interface compatible with Gymnasium
- Evaluation and monitoring tools

### 4.2 The PPO Algorithm (Proximal Policy Optimization)

PPO is an **Actor-Critic** algorithm that learns two things simultaneously:

1. **A Policy (Actor)**: A neural network that decides which action to take
   - Input: the observation (state)
   - Output: the probability distribution over actions

2. **A Value Function (Critic)**: A network that estimates the quality of a state
   - Input: the observation (state)
   - Output: the expected value of future rewards

**Why PPO is popular:**
- Stable: avoids overly aggressive updates through "clipping"
- Efficient: good performance/simplicity trade-off
- Robust: works well on a wide variety of problems

**Q5.** In an Actor-Critic algorithm like PPO, explain in your own words the role of the Actor and the Critic.

**Answer:**

**Q6.** Unlike supervised learning where we make a pass over a dataset, in RL training is done through **interaction loops**. Describe the steps of such a loop.

**Answer:**

---

## Part 5: Environment Implementation

### 5.1 The Glycemic Risk Function

The Kovatchev risk index is a clinical formula that:
- Is minimal (~0) around 112.5 mg/dL (optimal blood glucose)
- Penalizes hypoglycemia **more strongly** than hyperglycemia (asymmetry)
- Is used clinically to evaluate glycemic control

**Formula:**
```
f(BG) = 1.509 × (log(BG)^1.084 - 5.381)
risk = 10 × f(BG)²
```

**Q7.** Implement the `compute_risk_index(bg)` function that:
- Takes a blood glucose value `bg` in mg/dL
- Returns 100.0 if `bg <= 0` (invalid value)
- Otherwise computes the risk index using the formula above
- Clips the result to a maximum of 100.0

*Hint: use `np.log()` for the natural logarithm*

In [ ]:
# YOUR CODE HERE


**Q8.** Observe the risk values obtained. Verify that hypoglycemia (BG=50) has a higher risk than hyperglycemia (BG=180) even though both are at the same "distance" from the target zone. Why is this desirable from a medical standpoint?

**Answer:**

### 5.2 The Gymnasium Environment


**Gymnasium** (formerly OpenAI Gym) is a Python library that provides a standardized interface for Reinforcement Learning environments. It has become the de facto standard in the RL community.

**Why use Gymnasium?**
- Unified interface: all environments work the same way
- Compatibility: works with all RL libraries (Stable-Baselines3, RLlib, CleanRL, etc.)
- Large collection of ready-to-use environments (Atari, MuJoCo, etc.)

**Structure of a Gymnasium environment:**

An environment must define:

| Element | Description |
|---------|-------------|
| `observation_space` | Mathematical description of the observation space (type, dimensions, bounds) |
| `action_space` | Mathematical description of the action space (discrete, continuous, bounds) |
| `reset()` | Resets the environment and returns the initial observation |
| `step(action)` | Executes an action and returns (observation, reward, terminated, truncated, info) |

**Common space types:**
- `spaces.Discrete(n)`: Discrete actions (0, 1, 2, ..., n-1)
- `spaces.Box(low, high, shape)`: Continuous values within an interval
- `spaces.MultiDiscrete([n1, n2, ...])`: Multiple simultaneous discrete actions

**The standard interaction loop:**
```python
env = MyEnvironment()
obs, info = env.reset()

for _ in range(max_steps):
    action = agent.choose_action(obs)  # The agent chooses an action
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        break
```

Read the environment code below carefully. Identify:
- The dimension of the observation space
- The range of the action space
- How the action [-1, 1] is converted to an insulin dose

In [ ]:
class GlucoseEnv(gym.Env):
    """
    Glycemic control environment for Reinforcement Learning.

    Observation: 16-dimensional vector
        - 8 blood glucose values at different time horizons (normalized /400)
        - 8 corresponding insulin doses (normalized /0.1)

    Action: Continuous value in [-1, 1]
        - Converted to insulin dose: insulin = 0.05 + action * 0.05
        - So insulin ∈ [0, 0.1] U/min

    Reward: Risk difference = risk(t-1) - risk(t)
        - Positive if the situation improves
        - Negative if it worsens
    """

    def __init__(self, seed=None, patient_name='adult#001'):
        super().__init__()
        self.patient_name = patient_name
        self.seed_value = seed if seed is not None else np.random.randint(0, 100000)

        # Time horizons: T, T-3min, T-6min, T-15min, T-30min, T-1h, T-2h, T-4h
        # (in number of 3-minute steps)
        self.time_offsets = [0, 1, 2, 5, 10, 20, 40, 80]
        self.history_len = 81  # Keep enough history

        # Observation space: 8 BG + 8 insulin = 16 dimensions
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(16,), dtype=np.float32
        )

        # Action space: a continuous value between -1 and 1
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(1,), dtype=np.float32
        )

        # Histories
        self.bg_history = deque(maxlen=self.history_len)
        self.insulin_history = deque(maxlen=self.history_len)

        # Simulation parameters
        self.max_steps = 480  # 480 steps × 3 min = 24h
        self.prev_risk = 0.0

    def _create_env(self):
        """Creates the internal simglucose environment."""
        self.patient = T1DPatient.withName(self.patient_name)
        self.sensor = CGMSensor.withName('Dexcom', seed=self.seed_value)
        self.pump = InsulinPump.withName('Insulet')
        start_time = datetime.combine(datetime.now().date(), datetime.min.time())
        self.scenario = RandomScenario(start_time=start_time, seed=self.seed_value)
        self.internal_env = T1DSimEnv(self.patient, self.sensor, self.pump, self.scenario)

    def _get_bg(self, obs):
        """Extracts blood glucose from the simglucose observation."""
        return float(obs.CGM) if hasattr(obs, 'CGM') else float(obs)

    def _get_obs(self):
        """Builds the normalized observation vector."""
        obs = []
        for offset in self.time_offsets:
            idx = len(self.bg_history) - 1 - offset
            if idx >= 0:
                bg = self.bg_history[idx] / 400.0  # BG normalization
                ins = self.insulin_history[idx] / 0.1  # Insulin normalization
            else:
                # Padding if insufficient history
                bg = self.bg_history[0] / 400.0 if self.bg_history else 0.35
                ins = 0.5
            obs.extend([np.clip(bg, 0, 1), np.clip(ins, 0, 1)])
        return np.array(obs, dtype=np.float32)

    def reset(self, seed=None, options=None):
        """Resets the environment for a new episode."""
        if seed is not None:
            self.seed_value = seed

        self._create_env()
        self.step_count = 0
        self.bg_history.clear()
        self.insulin_history.clear()

        # First step of the simulator
        raw_obs, _, _, _ = self.internal_env.reset()
        bg = self._get_bg(raw_obs)
        self.bg_history.append(bg)
        self.insulin_history.append(0.05)  # Initial basal dose
        self.prev_risk = compute_risk_index(bg)

        return self._get_obs(), {'bg': bg}

    def step(self, action):
        """Executes an action and returns the result."""
        self.step_count += 1

        # Convert action [-1, 1] → insulin [0, 0.1]
        action_clipped = float(np.clip(action[0], -1, 1))
        insulin = 0.05 + action_clipped * 0.05

        # Execute in simglucose
        sim_action = Action(basal=insulin, bolus=0)
        raw_obs, _, _, _ = self.internal_env.step(sim_action)
        bg = self._get_bg(raw_obs)

        # Update histories
        self.bg_history.append(bg)
        self.insulin_history.append(insulin)

        # Compute reward: risk improvement
        current_risk = compute_risk_index(bg)
        reward = self.prev_risk - current_risk
        self.prev_risk = current_risk

        # Termination conditions
        terminated = bg < 40 or bg > 500  # Dangerous values
        truncated = self.step_count >= self.max_steps  # End of day

        info = {'bg': bg, 'insulin': insulin, 'risk': current_risk}

        return self._get_obs(), reward, terminated, truncated, info

print("✅ GlucoseEnv environment defined!")

**Q9.** Answer the following questions about the environment:

a) What is the dimension of the observation space?

b) If the agent chooses action = 0.5, what insulin dose will be injected?

c) How long (in hours) does a complete episode last?

**Answers:**

### 5.3 Testing the Environment

**Q10.** Create an instance of the environment with `seed=42` and `patient_name='adult#001'`. Then:
1. Perform a reset and display the initial blood glucose
2. Execute 10 steps with random actions
3. Display at each step: the step number, blood glucose, risk, and reward

In [ ]:
# YOUR CODE HERE


---

## Part 6: Training the PPO Agent

### 6.1 Model Configuration

We will create a PPO agent with the following hyperparameters:

| Hyperparameter | Value | Description |
|----------------|-------|-------------|
| `learning_rate` | 5e-4 | Learning rate |
| `n_steps` | 1024 | Number of steps before update |
| `batch_size` | 64 | Mini-batch size |
| `n_epochs` | 10 | Number of passes over the data |
| `gamma` | 0.95 | Discount factor (importance of the future) |
| `clip_range` | 0.2 | Policy change limit |
| `vf_coef` | 0.001 | Critic loss weight |
| `net_arch` | [32, 32] | Network architecture (2 layers of 32) |

**Q15.** The `gamma` parameter (discount factor) controls the importance given to future rewards relative to immediate rewards. If `gamma = 0`, the agent only considers the immediate reward. If `gamma = 1`, all future rewards have equal importance.

Why do we use `gamma = 0.95` rather than `gamma = 1` for glycemic control?

**Answer:**

**Q16.** Create the PPO model with the parameters from the table above.

📖 Documentation: https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html

Parameters to use:
- The previously created environment
- The hyperparameters from the table above
- A network architecture with **2 hidden layers of 32 neurons each** with tanh activation (use MlpPolicy)
- `verbose=1` to see training logs

*Hint: Check the documentation to find which policy type to use and how to specify the network architecture via `policy_kwargs`.*

In [ ]:
# YOUR CODE HERE


**Q17.** Observe the displayed model architecture. Identify:
- The input size of the network
- The layers of the policy_net (Actor)
- The layers of the value_net (Critic)
- The output size of the action_net

**Answer:**

In [ ]:
# YOUR CODE HERE


In [ ]:
# YOUR CODE HERE


### 6.2 Training

**Q18.** Launch training for **150,000 timesteps** with a progress bar.

⏱️ **Estimated time**: 10-15 minutes on Google Colab

*Note: During training, observe the metrics displayed. We will analyze them afterwards.*

In [ ]:
# YOUR CODE HERE


---

## Part 7: Understanding Training Metrics

During training, Stable-Baselines3 displays metrics. Here is an example of what you might see:

```
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 480          |
|    ep_rew_mean          | 0.601        |
| time/                   |              |
|    fps                  | 188          |
|    iterations           | 96           |
|    time_elapsed         | 522          |
|    total_timesteps      | 98304        |
| train/                  |              |
|    approx_kl            | 0.0018427707 |
|    clip_fraction        | 0.0249       |
|    clip_range           | 0.2          |
|    explained_variance   | 0.661        |
|    learning_rate        | 0.0005       |
|    loss                 | -0.0157      |
|    n_updates            | 950          |
|    policy_gradient_loss | -0.00227     |
|    std                  | 0.756        |
|    value_loss           | 0.238        |
------------------------------------------
```

**Q19.** For each metric below, look up its meaning and explain what it indicates about the learning process:

a) `ep_rew_mean` (mean episode reward)

b) `ep_len_mean` (mean episode length)

c) `explained_variance` 

d) `value_loss` (value function loss)

e) `clip_fraction`

**Answers:**

---

## Part 8: Agent Evaluation

### 8.1 Testing Predictions

**Q21.** Let's test the logic learned by the agent. For different blood glucose levels (80, 120, 180, 300 mg/dL), create an observation with that constant blood glucose across all time horizons and observe the predicted insulin dose.

To create the observation: `obs = np.array([bg/400, 0.5] * 8, dtype=np.float32)`

(Normalized BG + average insulin across the 8 time horizons)

Use `model.predict(obs, deterministic=True)` to get the action.

In [ ]:
# YOUR CODE HERE


**Q22.** Analyze the results. Has the agent learned a coherent logic? What does it do when blood glucose is low (80) versus high (300)?

**Answer:**

### 8.2 Comparison with a Random Policy

**Q23.** Implement a function `evaluate(policy_fn, name, n_episodes)` that:
1. Runs `n_episodes` episodes with the given policy function
2. For each episode, computes the TIR (Time In Range: % of time with 70 ≤ BG ≤ 180)
3. Displays the mean and standard deviation of the TIR and total reward

The function `policy_fn` takes an observation and returns an action.

In [ ]:
# YOUR CODE HERE


**Q24.** Evaluate two policies over 10 episodes each:
1. Random policy: `lambda obs: env.action_space.sample()`
2. PPO policy: `lambda obs: model.predict(obs, deterministic=True)[0]`

Compare the results.

In [ ]:
# YOUR CODE HERE


**Q25.** Interpret the results. Why is it important to compare with a random policy?

**Answer:**

---

## Part 9: Episode Visualization

**Q26.** Create a function `plot_episode(policy_fn, title)` that:
1. Runs a complete episode
2. Displays two side-by-side plots:
   - Blood glucose evolution with zones (hypoglycemia, target, hyperglycemia)
   - Insulin dose evolution

Use `fig, axes = plt.subplots(1, 2, figsize=(14, 5))` to create the two plots.

In [ ]:
# YOUR CODE HERE


**Q27.** Visualize one episode with the random policy and one episode with the PPO policy. Visually compare the results.

In [ ]:
# YOUR CODE HERE
